# Options from Zero · Episode 8: Maths pit stop: the bell curve in five minutes

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **The running trade.** An illustrative share priced at \$100 that pays no dividends, a one-year call option on it with a strike of \$105, 20% volatility and a 3.60% interest rate (continuously compounded). All made up for teaching; it is not market data.
>
> Educational material only, not financial advice. Options are risky: a seller can lose far more than the premium received.

1. Setup · 2. Last episode's question · 3. Coin flips make a bell · 4. Mean and spread · 5. The rule of thumb · 6. N(x) · 7. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Options from Zero",
    "episode": 8,
    "quantlib_version": ql.__version__,
    "quotes_label": "illustrative (made up for teaching)",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
S, K, T, r, vol = 100.0, 105.0, 1.0, 0.036, 0.20   # the running trade

The option formulas used throughout the series (`episodes/_shared/options.py`):

In [ ]:
# Option formulas for Options from Zero, written out by hand so every step can be read. Copied verbatim into
# each notebook by make_notebook.py, so every notebook runs on its own; the notebooks check these against
# QuantLib. Needs: math, numpy as np.
#   Running trade (Part 1): an illustrative share at $100, no dividends, 20% volatility, a one-year call struck
#   at $105, interest 3.60% a year continuously compounded. Made up for teaching.

def norm_cdf(x):
    """N(x): the chance a standard normal variable is below x."""
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def call_payoff(s, k):
    return np.maximum(np.asarray(s, dtype=float) - k, 0.0)

def put_payoff(s, k):
    return np.maximum(k - np.asarray(s, dtype=float), 0.0)

def black_scholes(s, k, t, r, vol, call=True):
    """Black-Scholes price of a European option on a share with no dividends (r continuously compounded)."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    if call:
        return s * norm_cdf(d1) - k * math.exp(-r * t) * norm_cdf(d2)
    return k * math.exp(-r * t) * norm_cdf(-d2) - s * norm_cdf(-d1)

def one_step(s, up, down, k, t, r, call=True):
    """One-step tree: replicate the option with delta shares and cash. Returns the pieces and the price."""
    pay = call_payoff if call else put_payoff
    v_up, v_down = float(pay(up, k)), float(pay(down, k))
    delta = (v_up - v_down) / (up - down)
    cash = (v_down - delta * down) * math.exp(-r * t)   # negative: borrowed today
    q = (s * math.exp(r * t) - down) / (up - down)       # risk-neutral probability of the up move
    return {"v_up": v_up, "v_down": v_down, "delta": delta, "cash": cash, "price": delta * s + cash, "q": q}

def binomial_tree(s, k, t, r, vol, steps, call=True, american=False, quantlib_p=False):
    """Cox-Ross-Rubinstein tree: u = exp(vol sqrt(dt)), d = 1/u, worked backward from expiry.
    quantlib_p=True uses QuantLib's up-probability 1/2 + 1/2 (r - vol^2/2) sqrt(dt) / vol instead of the textbook
    (e^{r dt} - d) / (u - d); both give the same price in the limit."""
    dt = t / steps
    u = math.exp(vol * math.sqrt(dt)); d = 1 / u
    p = 0.5 + 0.5 * (r - vol ** 2 / 2) * math.sqrt(dt) / vol if quantlib_p else (math.exp(r * dt) - d) / (u - d)
    pay = call_payoff if call else put_payoff
    prices = s * d ** np.arange(steps, -1, -1) * u ** np.arange(steps + 1)
    values = pay(prices, k)
    for _ in range(steps):
        prices = prices[1:] / u
        values = math.exp(-r * dt) * (p * values[1:] + (1 - p) * values[:-1])
        if american:
            values = np.maximum(values, pay(prices, k))
    return float(values[0])

## 2. Last episode's question

Why does the American put's exercise boundary rise as expiry nears? With less time left, the share has less chance to recover above the boundary, and the interest on \$105 received early is worth relatively more than waiting. The boundaries from Episode 7:

In [ ]:
out["answer"] = {"note": "the boundary rises as expiry nears; the numbers are in Episode 7"}

## 3. Coin flips make a bell

Flip a fair coin $n$ times and count heads. The exact chance of $k$ heads is $\binom{n}{k}/2^n$. As $n$ grows, the shape turns into the **normal distribution**, the bell curve, whatever the individual steps look like.

In [ ]:
def normal_pdf(x, mean, sd):
    return math.exp(-((x - mean) / sd) ** 2 / 2) / (sd * math.sqrt(2 * math.pi))

stages = {}
for n, width in ((10, 5), (100, 20), (1000, 60)):
    mean, sd = n / 2, math.sqrt(n) / 2
    ks = list(range(int(mean - width), int(mean + width) + 1))
    probs = [math.comb(n, k) / 2 ** n for k in ks]
    xs = np.linspace(mean - width, mean + width, 201)
    step = max(1, (2 * width) // 4)
    stages[str(n)] = {"n": n, "mean": mean, "sd": sd, "k": ks, "p": probs, "curve_x": xs.tolist(),
                      "curve_y": [normal_pdf(x, mean, sd) for x in xs], "x_range": [mean - width - 1, mean + width + 1],
                      "ticks": [{"x": float(x), "label": f"{x:.0f}"} for x in range(int(mean - width), int(mean + width) + 1, int(step))],
                      "max_gap": max(abs(p - normal_pdf(k, mean, sd)) for k, p in zip(ks, probs))}
out["flips"] = stages
{n: (s["mean"], round(s["sd"], 3), s["max_gap"]) for n, s in stages.items()}

## 4. Mean and spread

The **mean** is the centre: $n/2$ heads. The **standard deviation** measures the spread: for coin flips it is $\sqrt{n}/2$. Ten times more flips spread the count only $\sqrt{10} \approx 3.2$ times wider: spread grows with the square root of the number of steps.

In [ ]:
s100 = stages["100"]
out["spread"] = {"rows": [{"flips": s["n"], "mean": s["mean"], "sd": s["sd"]} for s in stages.values()],
                 "sd10": stages["10"]["sd"], "sd100": s100["sd"], "sd1000": stages["1000"]["sd"],
                 "lo100": s100["mean"] - s100["sd"], "hi100": s100["mean"] + s100["sd"],
                 "ratio": stages["1000"]["sd"] / s100["sd"],
                 "within1_exact": sum(p for k, p in zip(s100["k"], s100["p"]) if abs(k - 50) <= 5)}
pd.DataFrame(out["spread"]["rows"])

## 5. The rule of thumb

For any normal distribution: about 68% of outcomes lie within one standard deviation of the mean, 95% within two, 99.7% within three.

In [ ]:
out["rule"] = {"within1": norm_cdf(1) - norm_cdf(-1), "within2": norm_cdf(2) - norm_cdf(-2), "within3": norm_cdf(3) - norm_cdf(-3)}
out["rule"]["rows"] = [{"within": f"{k} sd", "share": v * 100} for k, v in zip((1, 2, 3), out["rule"].values())]
out["rule"]

## 6. N(x)

$N(x)$ is the chance that a standard normal outcome (mean 0, standard deviation 1) is below $x$: the area under the bell to the left of $x$. It's the $N$ in the Black-Scholes formula. Check against QuantLib's cumulative normal.

In [ ]:
xs = np.linspace(-3, 3, 121)
cn = ql.CumulativeNormalDistribution()
out["cdf"] = {"x": xs.round(3).tolist(), "y": [norm_cdf(x) for x in xs], "n1": norm_cdf(1), "n0": norm_cdf(0),
              "n_minus1": norm_cdf(-1), "ticks": [{"x": float(x), "label": f"{x:+.0f}" if x else "0"} for x in range(-3, 4)],
              "points_x": [0.0, 1.0], "points_y": [norm_cdf(0), norm_cdf(1)],
              "max_diff_quantlib": max(abs(norm_cdf(x) - cn(x)) for x in xs)}
assert out["cdf"]["max_diff_quantlib"] < 1e-12
out["question"] = {"flips": 400, "answer": math.sqrt(400) / 2}   # this episode's end-card question
{k: out["cdf"][k] for k in ("n1", "n0", "max_diff_quantlib")}

## 7. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")